# Opinion-Leader Message Production Mechanism Probe

- **Project:** Mass-communication opinion-leader model
- **Submodel ID and version:** `SM-OL-PROD-01`
- **Framework link:** Reviewed `opleader` rough design
- **Probe type:** Mechanism probe
- **Date:** 2026-09-04
- **Status:** Runnable

## 1. Question, Decision, and Framework Link

- **Primary question:** What production behavior follows when informed opinion leaders post with a constant probability but use either the Beta posterior mean or posterior-majority confidence to choose message stance?
- **Decision supported:** Retain a production null and a confidence-sensitive competitor for later researcher review before either is coupled to delivery and opinion updating.
- **Shared interfaces:** `AgentState`, `BetaBelief`, `Message`, `ProductionContext`, `ProductionOutcome`, and `MessageProduction`.
- **Highest intended claim level:** V1 isolated-mechanism behavior under fixed synthetic beliefs.


## 2. Provenance and Boundary Contract

- The two-stage separation between posting and stance is jointly developed and retained provisionally.
- Constant leader posting probability and all numerical factor levels are assumed experimental settings, not calibrated estimates.
- The posterior-mean stance rule is an assistant-proposed null. The confidence-sensitive competitor reuses the existing baseline rule.
- The probe uses an established-topic boundary: all declared leaders are already informed. Topic-knowledge activation is omitted.
- Ordinary agents are ineligible to produce. The external press produces one predetermined message in every configured round.
- Belief updating, delivery, network reach, aggregation, source-recipient weights, engagement, and all feedbacks are omitted.
- Each trial is an independent production opportunity from a fixed start-of-round state.


## 3. Expected Outcomes Before Running

- The press produces exactly one message with its scheduled stance in every configured round.
- Ordinary agents produce no messages and consume no random draws.
- Leader posting frequency depends on the constant posting probability, not on Beta mean or concentration.
- Under the posterior-mean rule, equal means imply equal stance probabilities at different concentrations.
- Under the confidence-sensitive rule, concentration makes a non-neutral mean more consistently express its favored side.
- The unconditional probability of a supportive leader message is the posting probability multiplied by the conditional support probability.
- Passing these checks supports implementation consistency only; it does not validate any empirical posting rate or expression mechanism.


In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'opinion_model').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from opinion_model.baseline import belief_from_mean_concentration
from opinion_model.core import AgentState, ProductionContext
from opinion_model.opleader import (
    LeaderMessageProduction,
    ScheduledPressMessageProduction,
    confidence_sensitive_support_probability,
    posterior_mean_support_probability,
)

RUN = {
    'submodel_id': 'SM-OL-PROD-01',
    'submodel_version': '0.1',
    'boundary_scenario': 'established topic with fixed one-round beliefs',
    'trials_per_condition': 5_000,
    'seed': 20260904,
    'leader_id': 1,
    'ordinary_id': 2,
    'press_id': 10,
    'press_stance': 1,
    'posting_probabilities': [0.0, 0.25, 0.5, 1.0],
    'belief_means': [0.4, 0.5, 0.6],
    'belief_concentrations': [4.0, 40.0],
    'active_feedbacks': [],
    'omitted_feedbacks': [
        'topic-knowledge activation', 'belief updating', 'delivery',
        'network reach', 'aggregation', 'engagement',
    ],
}
STANCE_RULES = {
    'posterior-mean': posterior_mean_support_probability,
    'confidence-sensitive': confidence_sensitive_support_probability,
}
RUN


## 4. Minimal Specification and Implementation

The notebook imports the tested mechanisms from `opinion_model.opleader`. For an eligible leader, posting is Bernoulli with the common probability `p_post`. Conditional on posting, stance is Bernoulli with probability `q_support` supplied by the selected stance rule. Thus the unconditional probability of a supportive message is `p_post * q_support`. The press is a deterministic external boundary provider rather than an adaptive agent.


## 5. Deterministic and Boundary Checks

These cases verify press determinism, ordinary-agent silence, posting-probability boundaries, neutral symmetry, and the distinct concentration behavior of the two stance rules.


In [ ]:
press_rounds = range(1, 6)
press = ScheduledPressMessageProduction(
    press_id=RUN['press_id'],
    stance_by_round={round_index: RUN['press_stance'] for round_index in press_rounds},
)
press_outcomes = tuple(press(round_index) for round_index in press_rounds)
assert all(outcome.did_post for outcome in press_outcomes)
assert all(outcome.post_probability == 1.0 for outcome in press_outcomes)
assert all(outcome.message.stance == RUN['press_stance'] for outcome in press_outcomes)

neutral_low = AgentState(belief_from_mean_concentration(0.5, 4.0))
neutral_high = AgentState(belief_from_mean_concentration(0.5, 40.0))
positive_low = AgentState(belief_from_mean_concentration(0.6, 4.0))
positive_high = AgentState(belief_from_mean_concentration(0.6, 40.0))
for rule in STANCE_RULES.values():
    assert math.isclose(rule(neutral_low), 0.5)
    assert math.isclose(rule(neutral_high), 0.5)
assert math.isclose(
    posterior_mean_support_probability(positive_low),
    posterior_mean_support_probability(positive_high),
)
assert (
    confidence_sensitive_support_probability(positive_high)
    > confidence_sensitive_support_probability(positive_low)
    > 0.5
)

leader_producer = LeaderMessageProduction(
    leader_ids=frozenset({RUN['leader_id']}),
    support_probability_rule=posterior_mean_support_probability,
)
ordinary_outcome = leader_producer(
    RUN['ordinary_id'], neutral_low, ProductionContext(1, 1.0),
    np.random.default_rng(1), np.random.default_rng(2),
)
zero_outcome = leader_producer(
    RUN['leader_id'], neutral_low, ProductionContext(1, 0.0),
    np.random.default_rng(3), np.random.default_rng(4),
)
unit_outcome = leader_producer(
    RUN['leader_id'], neutral_low, ProductionContext(1, 1.0),
    np.random.default_rng(5), np.random.default_rng(6),
)
assert not ordinary_outcome.did_post and ordinary_outcome.post_probability == 0.0
assert not zero_outcome.did_post and zero_outcome.message is None
assert unit_outcome.did_post and unit_outcome.message is not None

boundary_checks = pd.DataFrame([
    {
        'case': 'press fixed +',
        'did_post': press_outcomes[0].did_post,
        'post_probability': press_outcomes[0].post_probability,
        'support_probability': press_outcomes[0].support_probability,
    },
    {
        'case': 'ordinary ineligible',
        'did_post': ordinary_outcome.did_post,
        'post_probability': ordinary_outcome.post_probability,
        'support_probability': ordinary_outcome.support_probability,
    },
    {
        'case': 'leader p=0',
        'did_post': zero_outcome.did_post,
        'post_probability': zero_outcome.post_probability,
        'support_probability': zero_outcome.support_probability,
    },
    {
        'case': 'leader p=1',
        'did_post': unit_outcome.did_post,
        'post_probability': unit_outcome.post_probability,
        'support_probability': unit_outcome.support_probability,
    },
])
boundary_checks


## 6. Analytical Stance Comparison

This comparison evaluates each stance rule directly before adding posting stochasticity. It isolates whether Beta concentration changes expression at a fixed mean.


In [ ]:
analytical_rows = []
for mean in RUN['belief_means']:
    for concentration in RUN['belief_concentrations']:
        state = AgentState(belief_from_mean_concentration(mean, concentration))
        for rule_name, rule in STANCE_RULES.items():
            analytical_rows.append({
                'belief_mean': mean,
                'belief_concentration': concentration,
                'stance_rule': rule_name,
                'expected_support_given_post': rule(state),
            })
analytical = pd.DataFrame(analytical_rows)
mean_rule = analytical.query("stance_rule == 'posterior-mean'")
assert (
    mean_rule.groupby('belief_mean')['expected_support_given_post'].nunique() == 1
).all()
confidence_positive = analytical.query(
    "stance_rule == 'confidence-sensitive' and belief_mean == 0.6"
).sort_values('belief_concentration')
assert confidence_positive['expected_support_given_post'].is_monotonic_increasing
analytical.round(4)


## 7. Exploratory Stochastic Experiment

The factorial experiment crosses posting probability, Beta mean, Beta concentration, and stance rule. Each condition uses separate recorded posting and stance streams. Four-standard-error tolerances, with small numerical floors, are fixed before examining the simulated rates.


In [ ]:
def simulate_condition(post_probability, mean, concentration, rule_name, seed):
    state = AgentState(belief_from_mean_concentration(mean, concentration))
    rule = STANCE_RULES[rule_name]
    producer = LeaderMessageProduction(
        leader_ids=frozenset({RUN['leader_id']}),
        support_probability_rule=rule,
    )
    posting_rng = np.random.default_rng(seed)
    stance_rng = np.random.default_rng(seed + 1)
    post_count = 0
    support_count = 0
    for trial in range(RUN['trials_per_condition']):
        outcome = producer(
            RUN['leader_id'],
            state,
            ProductionContext(trial + 1, post_probability),
            posting_rng,
            stance_rng,
        )
        post_count += int(outcome.did_post)
        support_count += int(
            outcome.message is not None and outcome.message.stance == 1
        )

    trials = RUN['trials_per_condition']
    q_support = rule(state)
    observed_post_rate = post_count / trials
    observed_support_given_post = (
        support_count / post_count if post_count else math.nan
    )
    observed_support_unconditional = support_count / trials
    return {
        'post_probability': post_probability,
        'belief_mean': mean,
        'belief_concentration': concentration,
        'stance_rule': rule_name,
        'seed': seed,
        'trials': trials,
        'post_count': post_count,
        'support_count': support_count,
        'expected_post_rate': post_probability,
        'observed_post_rate': observed_post_rate,
        'expected_support_given_post': q_support,
        'observed_support_given_post': observed_support_given_post,
        'expected_support_unconditional': post_probability * q_support,
        'observed_support_unconditional': observed_support_unconditional,
    }

simulation_rows = []
condition_index = 0
for post_probability in RUN['posting_probabilities']:
    for mean in RUN['belief_means']:
        for concentration in RUN['belief_concentrations']:
            for rule_name in STANCE_RULES:
                condition_seed = RUN['seed'] + 2 * condition_index
                simulation_rows.append(simulate_condition(
                    post_probability, mean, concentration, rule_name, condition_seed
                ))
                condition_index += 1
results = pd.DataFrame(simulation_rows)

for row in results.itertuples(index=False):
    post_se = math.sqrt(
        row.expected_post_rate * (1.0 - row.expected_post_rate) / row.trials
    )
    assert abs(row.observed_post_rate - row.expected_post_rate) <= max(0.01, 4 * post_se)

    expected_unconditional = row.expected_support_unconditional
    unconditional_se = math.sqrt(
        expected_unconditional * (1.0 - expected_unconditional) / row.trials
    )
    assert (
        abs(row.observed_support_unconditional - expected_unconditional)
        <= max(0.015, 4 * unconditional_se)
    )

    if row.post_count:
        conditional_se = math.sqrt(
            row.expected_support_given_post
            * (1.0 - row.expected_support_given_post)
            / row.post_count
        )
        assert (
            abs(
                row.observed_support_given_post
                - row.expected_support_given_post
            )
            <= max(0.02, 4 * conditional_se)
        )
    else:
        assert row.expected_post_rate == 0.0
        assert math.isnan(row.observed_support_given_post)

results.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for (rule_name, concentration), group in analytical.groupby(
    ['stance_rule', 'belief_concentration']
):
    ordered = group.sort_values('belief_mean')
    axes[0].plot(
        ordered['belief_mean'],
        ordered['expected_support_given_post'],
        marker='o',
        label=f'{rule_name}, k={concentration:g}',
    )
axes[0].plot([0.4, 0.6], [0.4, 0.6], color='black', linewidth=0.8, linestyle=':')
axes[0].set(
    title='Conditional stance probability',
    xlabel='Beta mean',
    ylabel='P(support | post)',
)
axes[0].legend(frameon=False, fontsize=8)

axes[1].scatter(
    results['expected_support_unconditional'],
    results['observed_support_unconditional'],
    alpha=0.65,
)
limit = max(results['expected_support_unconditional'].max(), 0.01)
axes[1].plot([0, limit], [0, limit], color='black', linewidth=0.8, linestyle='--')
axes[1].set(
    title='Simulation against analytical rate',
    xlabel='Expected P(support message)',
    ylabel='Observed supportive-message rate',
)
fig.tight_layout()
plt.show()


## 8. Results and Conditional Interpretation

The next cell reports the deterministic and stochastic results. Interpretation remains conditional on fixed beliefs and independent production opportunities; it does not establish empirical validity.


In [ ]:
mean_positive = analytical.query(
    "stance_rule == 'posterior-mean' and belief_mean == 0.6"
).sort_values('belief_concentration')
confidence_positive = analytical.query(
    "stance_rule == 'confidence-sensitive' and belief_mean == 0.6"
).sort_values('belief_concentration')
representative = results[
    (results['post_probability'] == 0.5)
    & (results['belief_mean'] == 0.6)
    & (results['belief_concentration'] == 4.0)
    & (results['stance_rule'] == 'posterior-mean')
].iloc[0]
display(Markdown(
    f"""
- **Press and eligibility:** All `{len(press_outcomes)}` configured press rounds produced the fixed positive stance; the ordinary-agent boundary case remained silent.
- **Posterior-mean null:** At mean `0.6`, changing concentration from `{mean_positive.iloc[0]['belief_concentration']:.0f}` to `{mean_positive.iloc[-1]['belief_concentration']:.0f}` leaves conditional support probability at `{mean_positive.iloc[0]['expected_support_given_post']:.3f}`.
- **Confidence-sensitive competitor:** At the same mean, conditional support probability changes from `{confidence_positive.iloc[0]['expected_support_given_post']:.3f}` to `{confidence_positive.iloc[-1]['expected_support_given_post']:.3f}`.
- **Two-stage check:** For `p_post=0.5`, mean `0.6`, concentration `4`, and the posterior-mean rule, the expected unconditional supportive-message rate is `{representative['expected_support_unconditional']:.3f}` and the observed rate is `{representative['observed_support_unconditional']:.3f}`.
- **Highest completed level:** V1 isolated message-production behavior under fixed synthetic beliefs and an established-topic boundary.
- **Supports:** The implementation separates posting frequency from conditional stance and exposes the extra role of concentration in the confidence-sensitive rule.
- **Does not support:** Any empirical leader posting probability, Beta initialization, press stance, population message distribution, or integrated opinion-dynamics claim.
"""
))


## 9. Disposition and Change Impact

- **Disposition:** Retain the posterior-mean stance rule as the production null and the existing confidence-sensitive rule as a competing benchmark, pending researcher review.
- **Press boundary:** Retain deterministic externally scheduled press production for the first coupled case.
- **Affected mechanism:** `opleader` message production and this V1 notebook only.
- **Unaffected:** Baseline production behavior, source-recipient aggregation, shared state, scheduler, delivery, network, and platform mechanisms.
- **Deferred:** State-dependent posting, heterogeneous posting propensities, topic-knowledge activation, strategic expression, and ordinary-agent production.
- **Required later check:** V2 coupling only after the researcher selects the primary stance rule and the ownership and timing of `is_informed` are specified.
